In [ ]:
# OTHELLO bootstrap: make the package importable from notebooks/
import sys
from pathlib import Path
_repo_root = Path.cwd().parent.resolve()
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))


In [1]:
from huggingface_hub import login
import os
from dotenv import load_dotenv

# os.environ["TRANSFORMERS_CACHE"] = "/content/drive/Shareddrives/Algoverse_KSAC/hf_cache" #stores model
os.environ["HF_HOME"] = "../hf_home"  # stores logins

hf_token = os.getenv('HF_TOKEN')
# print(type(hf_token))
login(token=hf_token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [2]:
import os
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from pathlib import Path

variants = ['small', 'base', 'large']

paths = {variant:f'../models/google/flan-t5-{variant}' for variant in variants}

current_path =paths['small']
save_path = Path(current_path).resolve()

tokenizer = AutoTokenizer.from_pretrained(
    save_path,
    local_files_only=True
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    save_path,
    torch_dtype="auto",
    device_map="auto",
    local_files_only=True
)

print("Reloaded model successfully")
print(f"model.device = {model.device}")

`torch_dtype` is deprecated! Use `dtype` instead!


Reloaded model successfully
model.device = mps:0


#Dataloading (TODO)
#after sparqlgen + name parsing

In [6]:
import pandas as pd

In [26]:
names = ['mintaka', 'hotpot', 'qald']

dataframes = {name:pd.read_csv(f'../data/processed_names/{name}_processed.csv') for name in names}





In [8]:
# from transformers import pipeline
from transformers.generation.utils import GenerationMixin

In [27]:
print(dataframes['mintaka'].head())

   Unnamed: 0                                      AAVE Question  \
0           0  What da seventh tallest mountain in North Amer...   
1           1  Which actor was da star of Titanic and was bor...   
2           2  Which actor starred in Vanilla Sky and was mar...   
3           3  What year da first book of A Song of Ice and F...   
4           4               Who da youngest current US governor?   

                                        SAE Question  \
0  What is the seventh tallest mountain in North ...   
1  Which actor was the star of Titanic and was bo...   
2  Which actor starred in Vanilla Sky and was mar...   
3  In what year was the first book of the A Song ...   
4         Who is the youngest current U.S. governor?   

                                              SPARQL  \
0  SELECT ?item ?itemLabel ?elev WHERE {\n  ?item...   
1                                                NaN   
2                                                NaN   
3  SELECT (YEAR(?date) AS ?yea

In [ ]:
def answer(question, context):
    # Simpler, more direct prompt for FLAN
    prompt = f"""Context: {context}

Question: {question}

Answer the question ONLY using the context provided. If unsure, respond with 'I don't know'."""
    
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(model.device)
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        num_beams=4,
        early_stopping=True,
        do_sample=False,
    )
    
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
    
    return generated_text

# answered = {}
# for name, data in dataframes.items():
#     print(f'Starting processing for {name}')
#     answers = []
#     for index, row in data.iterrows():
#         question = row['SAE Question']
#         context = row['names']
#         llmanswer = answer(question, context)
#         answers.append(answer)
#         print(f'Row {index+1} processed, {llmanswer}')
#     data['Answer'] = answers
#     answered[name] = data

In [7]:
print(answer('What be da time zone of Salt Lake City?', None))

Central Time Zone


In [ ]:
import ast
import re

def clean_text(text):
    # If the entry is actually a list, flatten it
    if isinstance(text, list):
        text = text[0]
    elif isinstance(text, str):
       #If looks like a list []
        if text.strip().startswith('[') and text.strip().endswith(']'):
            try:
                parsed = ast.literal_eval(text)
                if isinstance(parsed, list) and len(parsed) > 0:
                    text = parsed[0]
            except Exception:
                pass

    # Now clean as before
    if not isinstance(text, str):
        return text

    text = re.sub(r'^(AAVE|SAE)\s*Question[:\s-]*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'^[\s\[\]\'"]+|[\s\[\]\'"]+$', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text



for name, data in answered.items():
# Apply to both columns
  data['SAE Question'] = data['SAE Question'].apply(clean_text)
  data['AAVE Question'] = data['AAVE Question'].apply(clean_text)
  file_path = f'../data/llm_answers/LLM_Answers_{name}.csv'
  data.to_csv(file_path, index=False)
  print(f"Translations completed and saved to {file_path}")





Translations completed and saved to ./LLM_answers/LLM_Answers_mintaka.csv
Translations completed and saved to ./LLM_answers/LLM_Answers_hotpot.csv
Translations completed and saved to ./LLM_answers/LLM_Answers_qald.csv


: 

nan